<a href="https://colab.research.google.com/github/umaimafawad711-code/Loan_prediction/blob/main/week3/LoanPrediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Import Libraries


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder


%matplotlib inline

Load Data & Basic Exploration (EDA)

In [ ]:
df = pd.read_csv("dataset.csv")

print("Dataset Shape:", df.shape)
df.head()
df.info()
df.describe()

Bar Chart for Missing Values

In [ ]:

# Missing values
missing = df.isnull().sum()
missing[missing > 0].sort_values().plot(kind='bar')
plt.title("Missing Values Per Column")
plt.show()

print(df.isnull().sum())

Visual Representation Of Data

In [ ]:
# 1. Target Distribution
sns.countplot(x="Loan_Status", data=df)
plt.title("Loan Status Distribution (Y = Yes, N = No)")
plt.show()

# 2. Income Distribution (Histogram)
sns.histplot(df["ApplicantIncome"], bins=30, kde=True)
plt.title("Applicant Income Distribution")
plt.show()

# 3. Correlation Heatmap
numeric_df = df.select_dtypes(include=['int64', 'float64'])
sns.heatmap(numeric_df.corr(), annot=True, cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.show()

sns.countplot(x='Gender', hue='Loan_Status', data=df)

Drop Useless Columns


In [ ]:
# Loan_ID Drop
df = df.drop(columns=['Loan_ID'])

# Check missing values
print(df.isnull().sum())

# check Duplicates
print(df.duplicated().sum())
df = df.drop_duplicates()

Feature Engineering

In [ ]:
# Total Income
df['TotalIncome'] = df['ApplicantIncome'] + df['CoapplicantIncome']

# EMI (LoanAmount / Loan_Amount_Term)
df['EMI'] = df['LoanAmount'] / df['Loan_Amount_Term']

print(df[['ApplicantIncome', 'CoapplicantIncome', 'TotalIncome', 'LoanAmount', 'Loan_Amount_Term', 'EMI']].head())

In [ ]:
# X = Inputs
X = df.drop(columns=['Loan_Status'])

# y = Target
y = df['Loan_Status']

print("X shape:", X.shape)
print("y shape:", y.shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)
print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

In [ ]:
# Numeric or Categorical columns lists
numeric_features = ['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount',
                    'Loan_Amount_Term', 'TotalIncome', 'EMI']
categorical_features = ['Gender', 'Married', 'Dependents', 'Education',
                        'Self_Employed', 'Credit_History', 'Property_Area']

# Pipeline for Numeric
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

#Pipeline for Categorical
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# combine Both
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

# Fit_transform
X_train_processed = preprocessor.fit_transform(X_train)

# Transform TEST only
X_test_processed = preprocessor.transform(X_test)

In [ ]:
print("X_train_processed shape:", X_train_processed.shape)
print("X_test_processed shape:", X_test_processed.shape)


# Check for missing values
print("Missing values in X_train:", np.isnan(X_train_processed).sum())
print("Missing values in X_test:", np.isnan(X_test_processed).sum())

**Linear Regression** was not used because our target variable (Loan_Status) is categorical (Y/N). Linear Regression is used for predicting continuous numerical values (like price), not categories. Logistic Regression is the correct equivalent for classification

KNN (K-Nearest Neighbors) Train


In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt

k_values = [1, 3, 5, 7, 9, 11, 13, 15]
accuracies = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_processed, y_train)
    y_pred_knn = knn.predict(X_test_processed)
    accuracies.append(accuracy_score(y_test, y_pred_knn))

plt.plot(k_values, accuracies, marker='o')
plt.title('K Value vs Accuracy')
plt.xlabel('K')
plt.ylabel('Accuracy')
plt.show()

Decision Tree Train

In [ ]:
from sklearn.tree import DecisionTreeClassifier

depths = [2, 3, 5, 10, None]
for d in depths:
    dt = DecisionTreeClassifier(max_depth=d, random_state=42)
    dt.fit(X_train_processed, y_train)
    y_pred_dt = dt.predict(X_test_processed)
    print(f"Max Depth: {d}, Accuracy: {accuracy_score(y_test, y_pred_dt)}")

Logistic Regression Train

In [ ]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(random_state=42)
lr.fit(X_train_processed, y_train)

# 5 test rows ki probability dekho
probabilities = lr.predict_proba(X_test_processed[:5])
print("Probability of [Reject, Approve]:\n", probabilities)

Evaluate All Models

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
import pandas as pd

# ---------------------------------------------------------
# 1. KNN Model
# ---------------------------------------------------------
knn_best = KNeighborsClassifier(n_neighbors=5)
knn_best.fit(X_train_processed, y_train)
y_pred_knn = knn_best.predict(X_test_processed)

print("--- KNN (K=5) Classification Report ---")
print(classification_report(y_test, y_pred_knn))

# ---------------------------------------------------------
# 2. Decision Tree Model
# ---------------------------------------------------------
dt_best = DecisionTreeClassifier(max_depth=3, random_state=42)
dt_best.fit(X_train_processed, y_train)
y_pred_dt = dt_best.predict(X_test_processed)

print("\n--- Decision Tree (Depth=3) Classification Report ---")
print(classification_report(y_test, y_pred_dt))

# ---------------------------------------------------------
# 3. Logistic Regression Model
# ---------------------------------------------------------

lr_best = LogisticRegression(random_state=42)
lr_best.fit(X_train_processed, y_train)
y_pred_lr = lr_best.predict(X_test_processed)

print("\n--- Logistic Regression Classification Report ---")
print(classification_report(y_test, y_pred_lr))

# ---------------------------------------------------------
# 4. Table
# ---------------------------------------------------------
results = {
    'Model': ['KNN (K=5)', 'Decision Tree (Depth=3)', 'Logistic Regression'],
    'Accuracy': [
        accuracy_score(y_test, y_pred_knn),
        accuracy_score(y_test, y_pred_dt),
        accuracy_score(y_test, y_pred_lr)
    ],
    'Precision (Y)': [
        precision_score(y_test, y_pred_knn, pos_label='Y'),
        precision_score(y_test, y_pred_dt, pos_label='Y'),
        precision_score(y_test, y_pred_lr, pos_label='Y')
    ],
    'Recall (Y)': [
        recall_score(y_test, y_pred_knn, pos_label='Y'),
        recall_score(y_test, y_pred_dt, pos_label='Y'),
        recall_score(y_test, y_pred_lr, pos_label='Y')
    ],
    'F1-Score (Y)': [
        f1_score(y_test, y_pred_knn, pos_label='Y'),
        f1_score(y_test, y_pred_dt, pos_label='Y'),
        f1_score(y_test, y_pred_lr, pos_label='Y')
    ]
}

# Print table
df_results = pd.DataFrame(results)
print("\n--- FINAL COMPARISON TABLE ---")

print(df_results.round(3))

**Confusion Matrix Of Best Model**
Based on the results, Logistic Regression is chosen as the best model. Although Decision Tree had slightly higher accuracy (0.845), Logistic Regression provides probabilities which are highly useful for banking. Also, in a banking problem, a False Positive (approving a defaulter) is very dangerous, so we need a model with high Recall and F1-score, which Logistic Regression provides (Recall 0.98)

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns


y_pred_best_knn = lr_best.predict(X_test_processed)
print(classification_report(y_test, y_pred_best_knn))

# Confusion Matrix Heatmap
cm = confusion_matrix(y_test, y_pred_best_knn)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()